# CITY RUNNER — LOCAL PARALLEL VERSION

This runner preserves the existing `load`, `single`, `bulk`, and `both` modes. Bulk mode assigns one scenario to each process; every process runs that scenario's seeds sequentially. Only the parent process writes Excel files.

Use a freshly restarted kernel before running this notebook so the numerical-library thread limits take effect.

In [1]:
# Run this cell first, immediately after restarting the kernel.
import os

for variable in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "NUMBA_NUM_THREADS",
):
    os.environ[variable] = "1"

import sys
from pathlib import Path
from IPython.display import display


def find_project_root(start_path=None):
    """Find the nearest folder containing the importable parallel core."""
    start = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "LLRP_CORE_PARALLEL.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate LLRP_CORE_PARALLEL.py. Place this notebook and the "
        "parallel core in the LLRP_GRASP+VND_Pharma_2 project root."
    )


PROJECT_ROOT = find_project_root()
BASE_DATA_PATH = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import LLRP_CORE_PARALLEL as llrp

print(f"Project root: {PROJECT_ROOT}")
print(f"Parallel core: {Path(llrp.__file__).name}")
print(f"Notebook Python: {sys.executable}")
print(f"Logical processors visible: {os.cpu_count()}")

LLRP_CORE_PARALLEL loaded. Use run_city_parallel(CITY_NAME, mode=...).
Project root: C:\Users\User\PycharmProjects\LLRP_GRASP+VND_Pharma_2
Parallel core: LLRP_CORE_PARALLEL.py
Notebook Python: C:\Users\User\PycharmProjects\LLRP_GRASP+VND_Pharma_2\.venv\Scripts\python.exe
Logical processors visible: 16


## Configuration

For the first parallel check, use `RUN_MODE = "bulk"`, `BULK_PROFILE = "validation"`, and `WORKERS = 1`. Run it once, then change only `WORKERS` to `2` and run again. The comparison cell below checks that solution outputs match.

Profiles:
- `validation`: scenarios 1–4, two seeds
- `benchmark`: scenarios 1–8, two seeds; test 4, 6, and 8 workers
- `full`: all 52 scenarios and five seeds

In [2]:
# ============================================================
# EDIT THESE VALUES
# ============================================================
CITY_NAME = "Elazig"
RUN_MODE = "bulk"  # "load", "single", "bulk", or "both"

BULK_PROFILE = "full"  # "validation", "benchmark", or "full"
WORKERS = 10                    # validation: 1 then 2; full: start with 4

RUN_OPTIONS = {
    "cost_per_km": 1.0,
    "max_route_duration": None,
    "max_iter": 50,
    "alpha_max": 5,
}

RUN_MODE = RUN_MODE.strip().lower()
BULK_PROFILE = BULK_PROFILE.strip().lower()
if RUN_MODE not in {"load", "single", "bulk", "both"}:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE!r}")
if BULK_PROFILE not in {"validation", "benchmark", "full"}:
    raise ValueError(f"Unsupported BULK_PROFILE: {BULK_PROFILE!r}")
if int(WORKERS) < 1:
    raise ValueError("WORKERS must be at least 1.")

if RUN_MODE in {"single", "both"}:
    RUN_OPTIONS["runs"] = 5
    RUN_OPTIONS["single_output_file"] = str(
        PROJECT_ROOT / CITY_NAME /
        f"Analysis_Report_{CITY_NAME}_VNDZouSO_CandidateDepots_PARALLEL_RUNNER.xlsx"
    )

if RUN_MODE in {"bulk", "both"}:
    if BULK_PROFILE == "validation":
        scenario_ids = [1, 2, 3, 4]
        run_seeds = [0, 1]
    elif BULK_PROFILE == "benchmark":
        scenario_ids = list(range(1, 9))
        run_seeds = [0, 1]
    else:
        scenario_ids = None
        run_seeds = [0, 1, 2, 3, 4]

    run_tag = f"{BULK_PROFILE.upper()}_W{int(WORKERS)}"
    city_folder = PROJECT_ROOT / CITY_NAME
    bulk_output = city_folder / f"{CITY_NAME}_Bulk_{run_tag}.xlsx"
    checkpoint_output = city_folder / f"{CITY_NAME}_Bulk_{run_tag}_checkpoint.xlsx"

    RUN_OPTIONS.update({
        "run_seeds": run_seeds,
        "scenario_ids": scenario_ids,
        "max_workers": int(WORKERS),
        "checkpoint_every": 1,
        "resume": True,
        "bulk_output_file": str(bulk_output),
        "checkpoint_file": str(checkpoint_output),
    })

print(f"Mode: {RUN_MODE}")
if RUN_MODE in {"bulk", "both"}:
    print(f"Bulk profile: {BULK_PROFILE}")
    print(f"Workers: {WORKERS}")
    print(f"Scenario IDs: {scenario_ids if scenario_ids else 'all 52'}")
    print(f"Seeds: {run_seeds}")
    print(f"Output: {bulk_output.name}")

Mode: bulk
Bulk profile: full
Workers: 10
Scenario IDs: all 52
Seeds: [0, 1, 2, 3, 4]
Output: Elazig_Bulk_FULL_W10.xlsx


In [3]:
import time

wall_start = time.perf_counter()
RESULT = llrp.run_city_parallel(
    city_name=CITY_NAME,
    mode=RUN_MODE,
    base_path=BASE_DATA_PATH,
    **RUN_OPTIONS,
)
TOTAL_WALL_TIME_S = time.perf_counter() - wall_start

print(f"Total notebook wall time: {TOTAL_WALL_TIME_S:.2f} seconds")
if RUN_MODE == "load":
    display(RESULT["candidate_depots"])
elif RUN_MODE == "single":
    display(RESULT["single"]["summary"])
elif RUN_MODE == "bulk":
    display(RESULT["bulk"]["scenario_summary"])
elif RUN_MODE == "both":
    display(RESULT["single"]["summary"])
    display(RESULT["bulk"]["scenario_summary"])

Loading real data for city: Elazig
Loading files:
 -> Index, parameters, depots: Elazig_index_map.xlsx
 -> Real road distance: Elazig_distance_matrix_Google.xlsx
 -> Real road duration: Elazig_duration_matrix_Google.xlsx
 -> [INFO] Priority column missing; using the explicitly labelled Equal-Priority baseline (p_j = 1.0).
Customer sheet: Sheet1
Candidate sheet: Sheet3 (header row 9)
Loaded customers (n): 69
Selected candidate depots (m): 7
Required open depots (Ng): 2
Maximum vehicles/routes (Nv): 8
Vehicle capacity (Q): 150
Fixed capacity per depot (W): 625
Total shift demand (D): 1055
Maximum route duration: 10000 seconds (city override)
Financial travel cost: distance_km × 1
Latency and route limit: real Google duration (seconds)
Candidate depot customer codes:
 -> D01: 10-1200301716
 -> D02: 10-1200301669
 -> D03: 10-1200301239
 -> D04: 10-1200301672
 -> D05: 10-1200301465
 -> D06: 10-1200301431
 -> D07: 10-1200301482
ELAZIG PARALLEL TWO-BLOCK BULK EXPERIMENT
n=69, m=7, Ng=2, Nv=8


,Scenario_ID,Scenario,Experiment_Block,Depot_Capacity_Level,Vehicle_Capacity_Level,Depot_Cap_W,Vehicle_Cap_Q,Alpha_Weight,Priority_Profile,Feasible_Runs,...,Std_Total_Cost,Best_Total_Cost,Worst_Total_Cost,Avg_Financial_Cost,Avg_Equity_Penalty,Avg_Num_SWH_Opened,Avg_Num_Routes,Avg_CPU_Time_s,Feasibility_Rate,Coverage_OK_Rate
0,1,S01_W560_Q135_A0_EQ,Capacity,Low,Low,560.0,135.0,0.0,Equal-Priority,5,...,3.5490,11675.83,11684.93,11679.8780,57352.4000,2.0,8.0,647.864,1.0,1.0
1,2,S02_W560_Q135_A0.5_EQ,Capacity,Low,Low,560.0,135.0,0.5,Equal-Priority,4,...,2418.4639,30184.98,35589.49,11706.5150,40739.0000,2.0,8.0,690.890,0.8,0.8
2,3,S03_W560_Q135_A1_EQ,Capacity,Low,Low,560.0,135.0,1.0,Equal-Priority,4,...,2092.8056,49688.68,54168.69,11702.6600,39349.2500,2.0,8.0,717.024,0.8,0.8
3,4,S04_W560_Q135_A2_EQ,Capacity,Low,Low,560.0,135.0,2.0,Equal-Priority,5,...,11250.1011,85289.52,112759.27,11710.1620,40618.4000,2.0,8.0,693.104,1.0,1.0
4,5,S05_W560_Q150_A0_EQ,Capacity,Low,Moderate,560.0,150.0,0.0,Equal-Priority,5,...,0.6129,11673.39,11675.06,11674.3820,57956.6000,2.0,8.0,665.832,1.0,1.0
5,6,S06_W560_Q150_A0.5_EQ,Capacity,Low,Moderate,560.0,150.0,0.5,Equal-Priority,5,...,71.4884,29552.80,29710.47,11704.0540,35867.0000,2.0,8.0,691.240,1.0,1.0
6,7,S07_W560_Q150_A1_EQ,Capacity,Low,Moderate,560.0,150.0,1.0,Equal-Priority,5,...,106.6403,47296.77,47552.86,11707.2960,35741.8000,2.0,8.0,681.768,1.0,1.0
7,8,S08_W560_Q150_A2_EQ,Capacity,Low,Moderate,560.0,150.0,2.0,Equal-Priority,5,...,294.0387,83159.27,83882.40,11706.4160,35909.4000,2.0,8.0,704.842,1.0,1.0
8,9,S09_W560_Q175_A0_EQ,Capacity,Low,High,560.0,175.0,0.0,Equal-Priority,5,...,1.0669,11471.56,11474.10,11472.9580,61704.4000,2.0,7.0,798.582,1.0,1.0
9,10,S10_W560_Q175_A0.5_EQ,Capacity,Low,High,560.0,175.0,0.5,Equal-Priority,5,...,79.9967,29458.65,29663.69,11702.6840,35706.2000,2.0,8.0,710.592,1.0,1.0


## Optional route map

This applies only after `single` or `both` mode.

In [ ]:
if RUN_MODE in {"single", "both"}:
    ROUTE_MAP = llrp.build_solution_map(RESULT["single"], use_osrm=True)
    display(ROUTE_MAP)
else:
    print("Run single mode first to create a route map.")